# Module 4: Train MobileNetV2 Surrogate

This notebook is the second split-out stage of Module 4. It trains the MobileNetV2 surrogate with centralized supervised learning on the attacker data view used for input-level adversarial examples and black-box transfer evaluation.

Run this notebook from the `4_Adversarial_FL/` directory.


## Stage Goal

Train a differentiable attacker model without using MobileNetV3 target gradients. The handoff artifact is `artifacts/module4_surrogate.pt`, which `attack_module.ipynb` loads to craft random-noise, FGSM, and PGD examples.


## 1. Notebook Setup

Load the surrogate-training utilities. This notebook uses ordinary supervised optimization and does not construct a federated server.


In [ ]:
from copy import deepcopy
from pathlib import Path

import json
import yaml
import numpy as np
import torch
import torchvision.transforms as tv_transforms
import matplotlib.pyplot as plt
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from model import MobileNetV2Transfer
from util_functions import evaluate_fn, set_seed
from load_data_for_clients import dist_data_per_client


## 2. Configuration and Artifact Setup

Load `config.yaml`, flatten the legacy and split surrogate-training settings into one config, resolve the device, and validate the supervised surrogate optimizer/scheduler settings.


In [ ]:
CONFIG_PATH = Path("config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Could not locate config.yaml in the working directory")
with CONFIG_PATH.open() as f:
    CONFIG = yaml.safe_load(f)


def _merge_dicts(base: dict | None, override: dict | None) -> dict:
    merged = deepcopy(base or {})
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = deepcopy(value)
    return merged


def _flatten_surrogate_training_config(config: dict) -> dict:
    cfg = _merge_dicts(config.get("surrogate", {}), config.get("surrogate_training", {}))
    optimizer_cfg = cfg.get("optimizer", {}) if isinstance(cfg.get("optimizer", {}), dict) else {}
    criterion_cfg = cfg.get("criterion", {}) if isinstance(cfg.get("criterion", {}), dict) else {}

    if "lr" in optimizer_cfg:
        cfg["learning_rate"] = optimizer_cfg["lr"]
        cfg["lr"] = optimizer_cfg["lr"]
    if "weight_decay" in optimizer_cfg:
        cfg["weight_decay"] = optimizer_cfg["weight_decay"]
    if "label_smoothing" in criterion_cfg:
        cfg["label_smoothing"] = criterion_cfg["label_smoothing"]
    return cfg


global_config = deepcopy(CONFIG.get("global_config", {}))
data_config = deepcopy(CONFIG.get("data_config", {}))
model_config = deepcopy(CONFIG.get("model_config", {}))
alg_configs = deepcopy(CONFIG.get("algorithms", {}))
artifact_config = deepcopy(CONFIG.get("artifacts", {}))
surrogate_training_config = _flatten_surrogate_training_config(CONFIG)
DEFAULT_FED_CONFIG = deepcopy(alg_configs.get("FedAvg", {}).get("fed_config", {}))
set_seed(global_config.get("seed", 42))


def get_device(preferred: str | None = None) -> torch.device:
    choice = preferred if preferred is not None else global_config.get("device")
    if isinstance(choice, str):
        if choice.startswith("cuda") and not torch.cuda.is_available():
            choice = "cpu"
        return torch.device(choice)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
global_config["device"] = str(DEVICE)

ARTIFACT_DIR = Path(artifact_config.get("dir", "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def artifact_path(key: str, default_filename: str) -> Path:
    return ARTIFACT_DIR / artifact_config.get(key, default_filename)


def validate_surrogate_config() -> None:
    issues = []
    optimizer_cfg = surrogate_training_config.get("optimizer", {}) if isinstance(surrogate_training_config.get("optimizer", {}), dict) else {}
    scheduler_cfg = surrogate_training_config.get("scheduler", {}) if isinstance(surrogate_training_config.get("scheduler", {}), dict) else {}
    optimizer_name = str(optimizer_cfg.get("name", "adamw")).lower()
    scheduler_name = str(scheduler_cfg.get("name", "none")).lower()
    num_epochs = int(surrogate_training_config.get("num_epochs", 0))
    batch_size = int(surrogate_training_config.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
    pool_size = int(surrogate_training_config.get("pool_size", 1))

    if num_epochs <= 0:
        issues.append("surrogate_training.num_epochs must be positive.")
    if batch_size <= 0:
        issues.append("surrogate_training.batch_size must be positive.")
    if pool_size <= 0:
        issues.append("surrogate_training.pool_size must be positive.")
    if optimizer_name not in {"adam", "adamw", "sgd"}:
        issues.append("surrogate_training.optimizer.name must be one of: adam, adamw, sgd.")
    if scheduler_name not in {"none", "cosine", "plateau", "reduce_on_plateau"}:
        issues.append("surrogate_training.scheduler.name must be one of: none, cosine, plateau, reduce_on_plateau.")
    if issues:
        raise ValueError("Surrogate-training config validation failed:\n" + "\n".join(issues))

    lr = optimizer_cfg.get("lr", surrogate_training_config.get("learning_rate", 1e-3))
    print(
        "Centralized surrogate config ready: "
        f"device={DEVICE}, dataset={surrogate_training_config.get('dataset_name', data_config.get('dataset_name'))}, "
        f"epochs={num_epochs}, batch_size={batch_size}, pool_size={pool_size}, "
        f"optimizer={optimizer_name}, lr={float(lr):.3g}"
    )


validate_surrogate_config()

config_snapshot = deepcopy(CONFIG)
config_snapshot.setdefault("global_config", {})["resolved_device"] = str(DEVICE)
config_snapshot.setdefault("surrogate_training", {})["resolved_mode"] = "centralized"
config_path = artifact_path("config_snapshot", "module4_config_used.json")
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

print("Loaded config from", CONFIG_PATH.resolve())
print(f"Saved config snapshot to {config_path.resolve()}")


## 3. Surrogate Data

Build a deterministic attacker data view by pooling one or more client shards from the same data distribution. The training loop is centralized: a single MobileNetV2 model trains with standard backpropagation over the pooled data.


In [ ]:
SURROGATE_CFG = surrogate_training_config
SURROGATE_CLIENT_ID = int(SURROGATE_CFG.get("client_id", 0))
SURROGATE_SEED = SURROGATE_CFG.get("seed", global_config.get("seed", 42))
SURROGATE_POOL_SIZE = max(1, int(SURROGATE_CFG.get("pool_size", 1)))
SURROGATE_NUM_CLIENTS = int(SURROGATE_CFG.get("num_clients", DEFAULT_FED_CONFIG.get("num_clients", 50)))
SURROGATE_BATCH_SIZE = int(SURROGATE_CFG.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))

_SURROGATE_CLIENT_LOADERS = None
_SURROGATE_TEST_LOADER = None
_SURROGATE_POOLED_LOADER = None


class AugmentedDataset(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, label = self.base_dataset[index]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def _surrogate_transform():
    if not SURROGATE_CFG.get("augment", False):
        return None
    return tv_transforms.Compose([
        tv_transforms.RandomHorizontalFlip(),
        tv_transforms.RandomRotation(degrees=10),
    ])


def _ensure_surrogate_data():
    global _SURROGATE_CLIENT_LOADERS, _SURROGATE_TEST_LOADER
    if _SURROGATE_CLIENT_LOADERS is not None and _SURROGATE_TEST_LOADER is not None:
        return _SURROGATE_CLIENT_LOADERS, _SURROGATE_TEST_LOADER

    loaders, test_loader = dist_data_per_client(
        data_path=SURROGATE_CFG.get("dataset_path", data_config.get("dataset_path")),
        dataset_name=SURROGATE_CFG.get("dataset_name", data_config.get("dataset_name")),
        num_clients=SURROGATE_NUM_CLIENTS,
        batch_size=SURROGATE_BATCH_SIZE,
        non_iid_per=SURROGATE_CFG.get("non_iid_per", data_config.get("non_iid_per", 0.0)),
        device=DEVICE,
    )
    _SURROGATE_CLIENT_LOADERS = loaders
    _SURROGATE_TEST_LOADER = test_loader
    return _SURROGATE_CLIENT_LOADERS, _SURROGATE_TEST_LOADER


def _build_surrogate_pool_loader():
    global _SURROGATE_POOLED_LOADER
    if _SURROGATE_POOLED_LOADER is not None:
        return _SURROGATE_POOLED_LOADER

    client_loaders, _ = _ensure_surrogate_data()
    pool_size = min(SURROGATE_POOL_SIZE, len(client_loaders))
    datasets = []
    for idx in range(pool_size):
        dataset = getattr(client_loaders[idx], "dataset", None)
        if dataset is None:
            raise ValueError("Client loader is missing a dataset attribute; cannot build surrogate pool")
        datasets.append(dataset)

    pooled_dataset = ConcatDataset(datasets)
    transform = _surrogate_transform()
    if transform is not None:
        pooled_dataset = AugmentedDataset(pooled_dataset, transform=transform)

    _SURROGATE_POOLED_LOADER = DataLoader(
        pooled_dataset,
        batch_size=SURROGATE_BATCH_SIZE,
        shuffle=True,
        num_workers=int(SURROGATE_CFG.get("num_workers", 0)),
        pin_memory=bool(SURROGATE_CFG.get("pin_memory", DEVICE.type == "cuda")),
    )
    return _SURROGATE_POOLED_LOADER


def get_surrogate_train_loader(pool: bool = True):
    client_loaders, _ = _ensure_surrogate_data()
    if not pool:
        if SURROGATE_CLIENT_ID >= len(client_loaders):
            raise IndexError(
                f"Surrogate client id {SURROGATE_CLIENT_ID} out of range for {len(client_loaders)} clients"
            )
        return client_loaders[SURROGATE_CLIENT_ID]
    return _build_surrogate_pool_loader()


def get_surrogate_test_loader():
    _, test_loader = _ensure_surrogate_data()
    return test_loader


surrogate_train_loader = get_surrogate_train_loader(pool=True)
surrogate_test_loader = get_surrogate_test_loader()
print(
    f"Surrogate data ready: train_examples={len(surrogate_train_loader.dataset)}, "
    f"test_examples={len(surrogate_test_loader.dataset)}"
)


## 4. Surrogate Training Helpers

Define the MobileNetV2 model, loss, optimizer, scheduler, and supervised training loop.


In [ ]:
def build_surrogate_model(num_classes: int = 10, pretrained: bool | None = None) -> torch.nn.Module:
    if pretrained is None:
        pretrained = SURROGATE_CFG.get("pretrained", True)
    return MobileNetV2Transfer(pretrained=pretrained, num_classes=num_classes)


def build_surrogate_criterion() -> torch.nn.Module:
    criterion_cfg = SURROGATE_CFG.get("criterion", {}) if isinstance(SURROGATE_CFG.get("criterion", {}), dict) else {}
    label_smoothing = float(criterion_cfg.get("label_smoothing", SURROGATE_CFG.get("label_smoothing", 0.0)))
    return torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing).to(DEVICE)


def build_surrogate_optimizer(model: torch.nn.Module) -> torch.optim.Optimizer:
    optimizer_cfg = SURROGATE_CFG.get("optimizer", {}) if isinstance(SURROGATE_CFG.get("optimizer", {}), dict) else {}
    name = str(optimizer_cfg.get("name", SURROGATE_CFG.get("optimizer_name", "adamw"))).lower()
    lr = float(optimizer_cfg.get("lr", SURROGATE_CFG.get("learning_rate", SURROGATE_CFG.get("lr", 1e-3))))
    weight_decay = float(optimizer_cfg.get("weight_decay", SURROGATE_CFG.get("weight_decay", 0.0)))
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError("No trainable parameters available for surrogate optimisation.")

    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        momentum = float(optimizer_cfg.get("momentum", SURROGATE_CFG.get("momentum", 0.9)))
        return torch.optim.SGD(trainable_params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(f"Unsupported surrogate optimizer: {name}")


def build_surrogate_scheduler(optimizer: torch.optim.Optimizer, epochs: int):
    scheduler_cfg = SURROGATE_CFG.get("scheduler", {}) if isinstance(SURROGATE_CFG.get("scheduler", {}), dict) else {}
    name = str(scheduler_cfg.get("name", "none")).lower()
    if name == "none":
        return None
    if name == "cosine":
        min_lr = float(scheduler_cfg.get("min_lr", scheduler_cfg.get("eta_min", 0.0)))
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(int(epochs), 1),
            eta_min=min_lr,
        )
    if name in {"plateau", "reduce_on_plateau"}:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=float(scheduler_cfg.get("factor", 0.5)),
            patience=int(scheduler_cfg.get("patience", 1)),
        )
    raise ValueError(f"Unsupported surrogate scheduler: {name}")


def _step_surrogate_scheduler(scheduler, val_loss: float) -> None:
    if scheduler is None:
        return
    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(val_loss)
    else:
        scheduler.step()


def train_surrogate_baseline(num_epochs: int | None = None):
    set_seed(global_config.get("seed", 42))
    train_loader = get_surrogate_train_loader(pool=True)
    test_loader = get_surrogate_test_loader()
    num_classes = SURROGATE_CFG.get("num_classes", model_config.get("kwargs", {}).get("num_classes", 10))
    model = build_surrogate_model(num_classes=num_classes).to(DEVICE)
    if SURROGATE_CFG.get("freeze_backbone", False) and hasattr(model, "v2model"):
        for param in model.v2model.features.parameters():
            param.requires_grad = False
    criterion = build_surrogate_criterion()
    optimizer = build_surrogate_optimizer(model)
    epochs = int(num_epochs or SURROGATE_CFG.get("num_epochs", 5))
    scheduler = build_surrogate_scheduler(optimizer, epochs)
    patience = int(SURROGATE_CFG.get("early_stop_patience", 0))
    use_amp = bool(SURROGATE_CFG.get("use_amp", False)) and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "lr": []}
    best_state = None
    best_val_loss = float("inf")
    epochs_since_improved = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE).float()
            labels = labels.to(DEVICE).long()
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            total += labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
        epoch_loss = running_loss / max(len(train_loader), 1)
        epoch_acc = 100.0 * correct / max(total, 1)
        val_loss, val_acc = evaluate_fn(test_loader, model, criterion, DEVICE)
        _step_surrogate_scheduler(scheduler, val_loss)
        current_lr = float(optimizer.param_groups[0]["lr"])
        history["loss"].append(float(epoch_loss))
        history["accuracy"].append(float(epoch_acc))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_acc))
        history["lr"].append(current_lr)
        print(
            f"Epoch {epoch + 1}/{epochs}: train_loss={epoch_loss:.4f}, train_acc={epoch_acc:.2f}%, "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.2f}%, lr={current_lr:.2e}"
        )
        if val_loss + 1e-5 < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_since_improved = 0
        else:
            epochs_since_improved += 1
            if patience and epochs_since_improved >= patience:
                print(f"Stopping early at epoch {epoch + 1} after {patience} epochs without improvement.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_acc = evaluate_fn(test_loader, model, criterion, DEVICE)
    summary = {
        "training_mode": "centralized",
        "model": "MobileNetV2Transfer",
        "dataset": SURROGATE_CFG.get("dataset_name", data_config.get("dataset_name")),
        "history": history,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "epochs_completed": len(history["loss"]),
        "pool_size": SURROGATE_POOL_SIZE,
        "batch_size": SURROGATE_BATCH_SIZE,
        "optimizer": deepcopy(SURROGATE_CFG.get("optimizer", {})),
        "scheduler": deepcopy(SURROGATE_CFG.get("scheduler", {})),
        "criterion": deepcopy(SURROGATE_CFG.get("criterion", {})),
        "use_amp": use_amp,
    }
    return model, summary


## 5. Train and Save the Surrogate

Train MobileNetV2 and check whether the held-out accuracy is high enough for gradient attacks to be meaningful.


In [ ]:
surrogate_model, surrogate_summary = train_surrogate_baseline()

print(
    f"Surrogate test metrics -> loss: {surrogate_summary['test_loss']:.4f}, "
    f"accuracy: {surrogate_summary['test_accuracy']:.2f}%"
)

surrogate_summary


### Save and Validate the Surrogate

Save `module4_surrogate.json` and the surrogate checkpoint, then run basic health checks on loss, epoch count, and test accuracy.


In [ ]:
surrogate_path = artifact_path("surrogate_metrics", "module4_surrogate.json")
with surrogate_path.open("w") as f:
    json.dump(surrogate_summary, f, indent=2)

surrogate_checkpoint_path = artifact_path("surrogate_checkpoint", "module4_surrogate.pt")
torch.save(surrogate_model.state_dict(), surrogate_checkpoint_path)

acc = float(surrogate_summary.get("test_accuracy", 0.0))
loss = float(surrogate_summary.get("test_loss", float("nan")))
history_len = int(surrogate_summary.get("epochs_completed", 0))
if not np.isfinite(loss):
    raise ValueError("Surrogate loss is not finite; revisit training settings.")
if history_len <= 0:
    raise ValueError("Surrogate training history is empty.")
if acc < 5.0:
    raise ValueError(f"Surrogate accuracy {acc:.2f}% is suspiciously low; revisit training settings.")
print(f"Surrogate test accuracy: {acc:.2f}%")
print(f"Saved details to {surrogate_path.resolve()}")
print(f"Saved checkpoint to {surrogate_checkpoint_path.resolve()}")


### Surrogate Training Curves

Plot the surrogate's training and validation curves. These curves help explain whether later transfer results come from a reasonable surrogate or from an underfit or overfit one.


In [ ]:
def plot_surrogate_history(summary: dict) -> None:
    history = summary.get("history", {})
    if not history.get("loss"):
        print("No surrogate history to plot.")
        return
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["loss"], marker="o", label="train")
    axes[0].plot(epochs, history.get("val_loss", []), marker="s", label="val")
    axes[0].set_title("Surrogate loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["accuracy"], marker="o", label="train")
    axes[1].plot(epochs, history.get("val_accuracy", []), marker="s", label="val")
    axes[1].set_title("Surrogate accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = artifact_path("surrogate_history_plot", "surrogate_history.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_surrogate_history(surrogate_summary)


## Handoff Artifacts

After this notebook runs, `attack_module.ipynb` expects these files in `artifacts/`:

| Artifact | Used by |
| --- | --- |
| `module4_surrogate.json` | Surrogate quality check before attacks |
| `module4_surrogate.pt` | Random-noise, FGSM, PGD, and transfer evaluation |
| `surrogate_history.png` | Instructor/student training-curve inspection |
